In [3]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("TIC 129839175") 
print(search_result)
lc2min = search_result[3].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

SearchResult containing 9 data products.

 #      mission     year   author  exptime target_name distance
                                      s                 arcsec 
--- --------------- ---- --------- ------- ----------- --------
  0  TESS Sector 31 2020      SPOC     120   129839175      0.0
  1  TESS Sector 97 2025      SPOC      20   129839175      0.0
  2  TESS Sector 97 2025      SPOC     120   129839175      0.0
  3 TESS Sector 105 2026      SPOC      20   129839175      0.0
  4 TESS Sector 105 2026      SPOC     120   129839175      0.0
  5  TESS Sector 31 2020 TESS-SPOC     600   129839175      0.0
  6  TESS Sector 31 2020       QLP     600   129839175      0.0
  7  TESS Sector 97 2025       QLP     200   129839175      0.0
  8  TESS Sector 31 2020      TARS     600   129839175      0.0


In [5]:
%matplotlib qt


plt.rcParams['font.size'] = '16' 
fig,ax = plt.subplots(1,1,figsize=(15,5))
plt.plot(x2min,y2min,'k')
labels= ["TIC 129839175"]
#ax.set_xlabel('Tempo - 2457000 [BTJD dias]').set_fontsize(16)
#ax.set_ylabel('Fluxo Normalizado').set_fontsize(16)  
#plt.title('Setor 27 (20s)').set_fontsize(17)
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)
plt.legend(labels) 
plt.show()

from IPython.display import Javascript, display

display(Javascript("alert('teste');"))

<IPython.core.display.Javascript object>

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.timeseries import BoxLeastSquares

%matplotlib qt
plt.rcParams['font.size'] = '16'

# ---------------- 0. Dados ----------------
t = np.asarray(x2min, float)
f = np.asarray(y2min, float)
ok = np.isfinite(t) & np.isfinite(f)
t, f = t[ok], f[ok]
s = np.argsort(t); t, f = t[s], f[s]
f = f / np.nanmedian(f)

# ---------------- 1. Parâmetros do catálogo (ExoFOP / TESS Project) ----------------
ALVO      = 'TIC 129839175'
P_CAT     = 0.64615709148349          # d      ± 1.56e-5
T0_CAT    = 2459144.977997 - 2457000  # BTJD   ± 3.44e-4  -> 2144.977997
DUR_EXP   = 1.1963822090877 / 24.0    # d      (1.196 h)
DEPTH_EXP = 6290.6211388837e-6        # ppm -> fração (± 96.7 ppm)
RP_RE     = 15.357                    # R_terra (± 8.06; muito mal restringido)
RS_RSUN   = 1.77                      # R_estrela <-- ajuste conforme o TIC

# ---------------- 2. Detrending: mediana móvel (janela >> duração, << período) ----------
JAN = 0.25                            # d (~5x a duração, ~0.39 P)
passo = JAN / 10
knots = np.arange(t.min(), t.max() + passo, passo)
i0 = np.searchsorted(t, knots - JAN/2, 'left')
i1 = np.searchsorted(t, knots + JAN/2, 'right')
mk = np.array([np.median(f[a:b]) if b - a > 10 else np.nan for a, b in zip(i0, i1)])
bom = np.isfinite(mk)
tend = np.interp(t, knots[bom], mk[bom])
fd = f / tend

# ---------------- 3. Clip de outliers (assimétrico: preserva o trânsito) --------------
med = np.median(fd)
sig = 1.4826 * np.median(np.abs(fd - med))
keep = (fd < med + 3.0*sig) & (fd > med - 10.0*sig)

tb, fb = t[keep], fd[keep]
print(f"Pontos usados: {tb.size} de {t.size}  |  sigma = {sig*1e6:.0f} ppm")
print(f"Baseline = {t.max()-t.min():.2f} d  ->  ~{(t.max()-t.min())/P_CAT:.0f} orbitas")

# ---------------- 4. BLS estreito em torno de 0.6462 d --------------------------------
bls  = BoxLeastSquares(tb, fb)
per  = np.linspace(0.60, 0.70, 40000)
durs = np.array([0.020, 0.030, 0.040, 0.050, 0.060, 0.075])
res  = bls.power(per, durs, objective='snr')

i   = int(np.argmax(res.power))
P   = res.period[i]
T0  = res.transit_time[i]
DUR = res.duration[i]
DEP = res.depth[i]

sde = (res.power[i] - np.median(res.power)) / (1.4826*np.median(np.abs(res.power-np.median(res.power))))
print(f"\nP   = {P:.6f} d      (catalogo: {P_CAT:.6f})")
print(f"T0  = {T0:.5f} BTJD   (catalogo: {T0_CAT:.5f})")
print(f"dur = {DUR*24:.2f} h    (esperado: {DUR_EXP*24:.2f} h)")
print(f"prof= {DEP*1e6:.0f} ppm  (esperado: {DEPTH_EXP*1e6:.0f} ppm)")
print(f"SDE = {sde:.1f}")

# fase do catalogo (propagada) vs BLS, em minutos
dfase = ((T0 - T0_CAT + 0.5*P_CAT) % P_CAT - 0.5*P_CAT) * 24 * 60
print(f"offset de fase BLS - catalogo = {dfase:+.1f} min")

stats = bls.compute_stats(P, DUR, T0)
d_odd, d_evn = stats['depth_odd'][0], stats['depth_even'][0]
e_odd, e_evn = stats['depth_odd'][1], stats['depth_even'][1]
sig_oe = abs(d_odd - d_evn) / np.hypot(e_odd, e_evn)
print(f"depth_odd/even = {d_odd*1e6:.0f} / {d_evn*1e6:.0f} ppm  ({sig_oe:.1f} sigma)")
print(f"N transitos com dados = {len(stats['transit_times'])}")

# secundario em fase 0.5
ph_all = (tb - T0) % P / P
insec  = np.abs(ph_all - 0.5) < (DUR/P)/2
base   = (np.abs(ph_all - 0.25) < (DUR/P)/2) | (np.abs(ph_all - 0.75) < (DUR/P)/2)
if insec.sum() > 10 and base.sum() > 10:
    dsec  = np.mean(fb[base]) - np.mean(fb[insec])
    edsec = np.hypot(np.std(fb[base])/np.sqrt(base.sum()), np.std(fb[insec])/np.sqrt(insec.sum()))
    print(f"secundario (fase 0.5) = {dsec*1e6:+.0f} +/- {edsec*1e6:.0f} ppm  ({dsec/edsec:+.1f} sigma)")

# ---------------- 5. Periodograma BLS -------------------------------------------------
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(res.period, res.power, 'k', lw=1)
ax.axvline(P_CAT, color='tab:red', ls='--', lw=2, label=f'catalogo {P_CAT:.5f} d')
ax.axvline(P, color='tab:blue', ls=':', lw=2, label=f'BLS {P:.5f} d')
ax.set_xlabel('Periodo [dias]', fontsize=16)
ax.set_ylabel('Potencia BLS (SNR)', fontsize=16)
ax.legend(fontsize=13)
plt.tight_layout()

# ---------------- 6. Curva de luz: setor inteiro + zoom de 2 d ------------------------
n0 = int(np.floor((t.min() - T0)/P)) - 1
n1 = int(np.ceil ((t.max() - T0)/P)) + 1
tc = T0 + P*np.arange(n0, n1+1)
tc = tc[(tc > t.min()) & (tc < t.max())]

fig, (a1, a2) = plt.subplots(2, 1, figsize=(15, 8))
a1.plot(t, f, 'k', lw=0.5, zorder=1)
a1.plot(t, tend, color='tab:orange', lw=1.5, zorder=2, label='tendencia (mediana movel)')
a1.set_ylabel('Fluxo Norm.', fontsize=16)
a1.set_title(f'{ALVO} (TOI 2440.01) - P = {P:.5f} d', fontsize=17)
a1.legend(fontsize=13)

t0z = tc[len(tc)//2] - 1.0
m = (t > t0z) & (t < t0z + 2.0)
a2.plot(t[m], fd[m], 'k.-', lw=0.6, ms=3, zorder=1)
for k, c in enumerate(tc[(tc > t0z) & (tc < t0z + 2.0)]):
    a2.axvspan(c-DUR/2, c+DUR/2, color='tab:red', alpha=0.25, zorder=0,
               label='transito previsto' if k == 0 else None)
    a2.axvline(c, color='tab:red', ls='--', lw=1.2, zorder=2)
a2.set_xlabel('Tempo - 2457000 [BTJD dias]', fontsize=16)
a2.set_ylabel('Fluxo Norm. (detrend)', fontsize=16)
a2.legend(fontsize=13)
plt.tight_layout()

# ---------------- 7. Fase dobrada: zoom no transito + fase completa -------------------
def binar(x, y, lo, hi, nb):
    ed  = np.linspace(lo, hi, nb+1)
    idx = np.digitize(x, ed) - 1
    xb = np.array([np.mean(x[idx == j]) if np.any(idx == j) else np.nan for j in range(nb)])
    yb = np.array([np.mean(y[idx == j]) if np.any(idx == j) else np.nan for j in range(nb)])
    eb = np.array([np.std(y[idx == j])/np.sqrt(np.sum(idx == j)) if np.sum(idx == j) > 1 else np.nan
                   for j in range(nb)])
    return xb, yb, eb

ph = (tb - T0 + 0.5*P) % P - 0.5*P

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 5))

sel = np.abs(ph) < 3*DUR
xb, yb, eb = binar(ph[sel], fb[sel], -3*DUR, 3*DUR, 90)
a1.plot(ph[sel]*24, fb[sel], '.', color='0.75', ms=2, zorder=1)
a1.errorbar(xb*24, yb, yerr=eb, fmt='o', color='k', ms=5, lw=1.2, zorder=2)
a1.axhline(1.0, color='0.4', lw=1)
a1.axhline(1.0 - DEPTH_EXP, color='tab:red', ls='--', lw=1.5,
           label=f'esperado ({DEPTH_EXP*1e6:.0f} ppm)')
a1.axvspan(-DUR_EXP/2*24, DUR_EXP/2*24, color='tab:blue', alpha=0.12)
a1.set_xlabel('Tempo desde o meio do transito [h]', fontsize=16)
a1.set_ylabel('Fluxo Normalizado', fontsize=16)
a1.set_title(f'T0 = {T0:.5f} BTJD', fontsize=16)
a1.legend(fontsize=12)

phf = (tb - T0) % P / P
xf, yf, ef = binar(phf, fb, 0, 1, 200)
a2.errorbar(xf, yf, yerr=ef, fmt='o', color='k', ms=3, lw=0.8)
a2.axhline(1.0, color='0.4', lw=1)
a2.axvline(0.5, color='tab:green', ls=':', lw=1.5, label='fase 0.5 (secundario)')
a2.set_xlabel('Fase', fontsize=16)
a2.set_title('Fase completa', fontsize=16)
a2.legend(fontsize=12)
plt.tight_layout()
plt.show()

Pontos usados: 88682 de 88814  |  sigma = 4074 ppm
Baseline = 25.17 d  ->  ~39 orbitas

P   = 0.646059 d      (catalogo: 0.646157)
T0  = 4208.16003 BTJD   (catalogo: 2144.97800)
dur = 0.72 h    (esperado: 1.20 h)
prof= 5004 ppm  (esperado: 6291 ppm)
SDE = 51.3
offset de fase BLS - catalogo = +3.5 min
depth_odd/even = 4882 / 5120 ppm  (0.0 sigma)
N transitos com dados = 39
secundario (fase 0.5) = +9 +/- 76 ppm  (+0.1 sigma)
